# 4.4 Outcome Verification on τ²-bench: Does a Graph Check of the State Catch Failed Actions Better Than an LLM?

*Formerly notebook 28.*

> **Outputs in this copy come from the confidence run** (`SMASH_MODE=batch`): 278 τ²-bench tasks × 3 passes, 18,780
> calls, $4.64; recording rounds from step 31 at the standard tier (2026-11-10), verifiers via the Batch API. Task
> 5.4. They are reported in `RESULTS-REPORT.md` and the v2 paper.

**Presentation map:** Task 4.4 (verification of outcomes against the graph), Task 4.3 (execution state; the recordings
made here are Notebook 4.3's reference runs). Paper v2, `PLAN.md` §5 row 5.4, §5a, §5b.

## Protocol (frozen before the confidence run)

**Research question.** When an agent's write action reports success, does checking the database state after the action
against the change expected for that tool, written as a graph pattern, detect failed or wrong actions better than an LLM
asked whether the action succeeded, from the tool output or from the same before/after state?

**Hypothesis.** H1: the graph check (3) detects failed actions with a higher F1 than the LLM given the same before/after
state (2). H2: the LLM that sees only the tool output and the conversation (1) misses most silent failures (recall near
0), because the tool output claims success; published work finds the same for implicit faults (ToolMaze 2026; Tools
Fail 2024).

**Data.** τ²-bench (Barres et al. 2025; `sierra-research/tau2-bench`, MIT), pinned at commit `96a3ef9` (the commit of
Notebook 3.2), base split: **airline 50 + retail 114 + telecom 114 = 278 tasks**. Each task is **recorded three times**
(passes with seeds 7, 8, 9; same temperature and prompts; pass 1 is Notebook 4.3's set of reference runs):
`gpt-6-luna` as the agent (τ²-bench's LLM-agent prompt with the domain policy and tools) and as the user simulator
(τ²-bench's user prompt and, in telecom, the user's device tools), temperature 0, τ²-bench's limits (100 steps, 10 tool
errors). Every conversation round is one `llm.run` over all live conversations, so in batch mode each round is one Batch
job (as Notebooks 1.1b and 1.2b). The environment (tools over the database, task initial state, evaluation) is τ²-bench's own
code, run unchanged in a separate venv (`examples/.venv-tau2`) behind a small JSON-lines server.

**Items.** Every **agent write call** in the recordings (a tool τ²-bench marks as state-mutating) that returned without
error and changed the database. An action that occurs in more than one pass with the same task, tool, arguments and
state of the records before the call is kept once (first pass). Each action is replayed **offline in up to five
variants**: **correct** (the real state after the call), **silent no-op** (the same success output, database
unchanged), and **three wrong changes**, each with the same success output: **wrong record** (the intended change lands
on another record of the same collection, of the same owner where possible), **wrong field** (one intended field stays
unchanged and another field of the record changes instead), **wrong value** (a changed field gets a different value,
e.g. a number shifted as if applied twice); a wrong-change kind that does not apply to the call is left out.
Positive class = failed action (no-op or wrong change). τ²-bench's gold-action replays are never used as items (the
patterns were checked on them).

**Verifiers** (all receive the tool call and the tool's description):
1. **LLM from the tool output**: the conversation up to the call, the call and its output.
2. **LLM with the state (same information as the graph check)**: the same, plus the **before and after records** that the
   call names or that changed (large maps such as flight dates are cut to the entries that differ or that the call names;
   identical for (2) and (3)).
3. **Graph check (code, no LLM)**: the records become a property graph (record = node, fields = properties, id
   references = edges); the expected change of each write tool is a **pattern written once per tool** (bound nodes,
   expected property changes, and a frame of nodes allowed to change), matched against the diff of the two graphs.
   **19 patterns, all hand-written** (one per write tool of the three domains).
   - **(3b) LLM-written patterns**: the same checker with patterns that `gpt-6-luna` writes once per tool from the tool's
     description and one sample record per collection (19 calls), without correction.

**Primary metric.** F1 of detecting failed actions, **graph check (3) − LLM with state (2)**, with a paired 95% bootstrap
interval that resamples **tasks** (5,000 resamples; all items of a task move together). **Secondary:** precision and
recall; recall per failure kind; false alarms on correct actions; per domain; (3b) vs. (3); (2) vs. (1); tokens and USD.

**n.** All distinct eligible write actions of the 3 × 278 recordings × up to 5 variants (the expected numbers are
printed by the forecast cell; see changes 2–4).

**Threats.** The faults are synthetic (injected into the recorded state, not real tool bugs) and were designed by the
author of the patterns; the LLM-written variant (3b) and the per-kind recall show how much depends on that. "Correct"
means the call executed as requested, not that it was the right call under the policy. The patterns were checked once on
replays of τ²-bench's gold actions (not on the recorded runs): one checker fix (list-valued arguments) and one pattern
change (`book_reservation` must change the named flights) before freezing. One model for agent, user and verifiers.

**Protocol changes after the test run:**
1. In the test run the user simulator (`gpt-6-luna`, no reasoning) solved 2 of 3 telecom tasks on its own device before
   the agent had said anything beyond its greeting (the agent made no call at all), which τ²-bench's user guidelines
   forbid ("Only call a tool if the agent has requested it"). The user's device tools are now given to the user
   simulator only after the agent's first reply; everything else is τ²-bench's prompt unchanged. (This changes only
   telecom conversations.)
2. The test run had 0.3 eligible agent write actions per task, not the ~5 the plan assumed (most τ²-bench tasks need
   one write, many none, and `gpt-6-luna` often transfers telecom customers to a human before writing). The full run is
   expected to give ≈ 100–150 write actions (≈ 300–450 variants), so the primary interval will probably be wider than
   the ±5-point target of PLAN §5b rule 3.
3. Therefore (coordinator's decision): **three recording passes** (seeds 7, 8, 9) instead of one; identical actions
   (task, tool, arguments, before-state) are deduplicated across passes; Notebook 4.3 keeps using pass 1 only.
4. Therefore also: **all three wrong-change kinds per action** instead of one seeded choice (1 correct + 1 no-op + up
   to 3 wrong variants per action). The bootstrap still resamples whole tasks (all passes and variants of a task move
   together). With this, failed variants outnumber correct ones about 4 : 1, so precision is higher than with 2 : 1 at
   the same false-alarm rate; the false-alarm rate on correct actions is reported next to it.

In [ ]:
# Notebooks live in examples/<task group>/; data/, smash_batch.py and the .venv-* environments stay in examples/.
# Every path below is relative to examples/, so run from there whether the notebook is opened in its own folder or not.
import os, sys
if not os.path.exists("smash_batch.py") and os.path.exists(os.path.join("..", "smash_batch.py")):
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [1]:
import json, os, re, random, subprocess, shutil, fnmatch, time
from collections import Counter, defaultdict
import numpy as np
import networkx as nx
from pydantic import BaseModel, Field
from smash_batch import LLM, mode_from_env, wilson

MODE = mode_from_env()               # replay (default) | test (small live run) | batch (full run via Batch API)
TEST = MODE == "test"
BASE = "data/benchmarks/tau2"
llm = LLM(f"{BASE}/cache" + ("_test" if TEST else ""))
TEST_TASKS = {"airline": 4, "retail": 3, "telecom": 3}   # test run: the first tasks of each domain, fixed order
print("MODE =", MODE)

MODE = batch


## τ²-bench: download (pinned commit) and its own environment

The repository is cloned once, sparse (code plus the airline, retail and telecom data, ~150 MB), at the commit Notebook 3.2
uses. τ²-bench needs Python ≥ 3.12, so it is installed into its own venv `examples/.venv-tau2` (`uv venv` + `pip install
-e`, plus `websockets`, which its package imports); the notebook's kernel is not changed. Licence: MIT (`LICENSE` in the
repository).

In [2]:
TAU2_COMMIT = "96a3ef92329baf908ffe9dfe188cea34e8701d2a"
REPO, VENV = f"{BASE}/tau2-bench", ".venv-tau2"
PYV = f"{VENV}/bin/python"
UV = shutil.which("uv") or os.path.expanduser("~/.local/bin/uv")
sh = lambda *c: subprocess.run(c, check=True)
if not os.path.exists(f"{REPO}/pyproject.toml"):
    sh("git", "clone", "--filter=blob:none", "--no-checkout", "https://github.com/sierra-research/tau2-bench.git", REPO)
    sh("git", "-C", REPO, "sparse-checkout", "set", "--no-cone", "/*", "!/data/", "!/web/", "!/figs/",
       "/data/tau2/domains/airline/", "/data/tau2/domains/retail/", "/data/tau2/domains/telecom/", "/data/tau2/user_simulator/")
    sh("git", "-C", REPO, "checkout", "-q", TAU2_COMMIT)
if not os.path.exists(PYV):
    sh(UV, "venv", "-q", "--python", "3.12", VENV)
    sh(UV, "pip", "install", "-q", "--python", PYV, "-e", REPO, "websockets")
head = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
ver = subprocess.run([PYV, "-c", "import sys; print(sys.version.split()[0])"], capture_output=True, text=True).stdout.strip()
print(f"tau2-bench at {head[:7]} (pinned {TAU2_COMMIT[:7]}), {VENV}: Python {ver}")
print(open(f"{REPO}/LICENSE").read().splitlines()[0])
assert head == TAU2_COMMIT

tau2-bench at 96a3ef9 (pinned 96a3ef9), .venv-tau2: Python 3.12.14
MIT License


### The environment server

A small JSON-lines server around τ²-bench's own environment classes (it is written to a file and run in the τ²-bench venv).
It is stateless: each request carries the tool calls already executed, and the environment is rebuilt by replaying them,
so replaying this notebook from its cache gives identical tool outputs. Notebook 4.3 uses the same server.

In [3]:
%%writefile data/benchmarks/tau2/tau2_env_server.py
"""tau2-bench environment server for notebooks 4.3 and 4.4 (written by notebook 4.4; do not edit by hand).

Runs inside examples/.venv-tau2 (tau2-bench needs Python >= 3.12) and speaks JSON lines on stdin/stdout: one request
{"op": ..., ...} per line, one response per line. It is stateless: every request carries the tool calls already
executed against the task's environment (`log`), and the environment is rebuilt by replaying them (a few ms), so the
notebook can be re-run from its LLM cache and gets identical tool outputs. Random ids (telecom bill ids) come from a
generator seeded per task, for the same reason.

Ops
  meta      -> agent system prompt and tools per domain, user-simulator prompt and tools per task
  exec      -> run new tool calls after replaying `log`
  evaluate  -> tau2-bench's state-based checks on the environment after `log` (DB hash vs. gold actions, env
               assertions) and its deterministic message checks (communicate, action); NL assertions (LLM-judged)
               are not run
  variants  -> for every successful agent write in `log`: the records before/after it, plus a silent no-op and
               every applicable wrong-change variant (wrong record, wrong field, wrong value) of the after state
  records   -> current content of given records after `log`
  named     -> the records a call's arguments name, after `log` (notebook 4.3's checkpoint verification)
"""
import copy
import json
import random
import sys
import uuid

_OUT = sys.stdout
sys.stdout = sys.stderr                       # tau2 prints/logs must not reach the protocol channel
from loguru import logger                     # noqa: E402

logger.remove()
from tau2.agent.llm_agent import AGENT_INSTRUCTION, SYSTEM_PROMPT  # noqa: E402
from tau2.data_model.message import AssistantMessage, ToolCall, ToolMessage, UserMessage  # noqa: E402
from tau2.environment.toolkit import get_tool_types  # noqa: E402
from tau2.evaluator.evaluator_action import ActionEvaluator  # noqa: E402
from tau2.evaluator.evaluator_communicate import CommunicateEvaluator  # noqa: E402
from tau2.registry import registry  # noqa: E402
from tau2.user.user_simulator import UserSimulator  # noqa: E402

DOMAINS = ("airline", "retail", "telecom")
_RNG = random.Random(0)
uuid.uuid4 = lambda: uuid.UUID(int=_RNG.getrandbits(128), version=4)   # deterministic ids on replay

TASKS = {d: {t.id: t for t in registry.get_tasks_loader(d)()} for d in DOMAINS}
ENVC = {d: registry.get_env_constructor(d) for d in DOMAINS}


def new_env(domain, task_id):
    task = TASKS[domain][task_id]
    _RNG.seed(f"{domain}|{task_id}")
    env = ENVC[domain]()
    st = task.initial_state
    env.set_state(initialization_data=st.initialization_data if st else None,
                  initialization_actions=st.initialization_actions if st else None,
                  message_history=list(st.message_history or []) if st else [])
    return env


def call(env, c):
    args = c.get("arguments")
    tc = ToolCall(id=c.get("id") or "call", name=c["name"], arguments=args if isinstance(args, dict) else {},
                  requestor=c.get("requestor", "assistant"))
    m = env.get_response(tc)
    return {"content": m.content, "error": bool(m.error)}


def replay(domain, task_id, log):
    env = new_env(domain, task_id)
    for c in log:
        call(env, c)
    return env


def sample(rec, k=2):
    """One record as an example of a collection's schema: long maps and lists cut to k entries."""
    if isinstance(rec, dict):
        items = list(rec.items())
        if len(items) > 8 and all(isinstance(v, dict) for _, v in items):
            items = items[:k]
        return {kk: sample(v, k) for kk, v in items}
    if isinstance(rec, list):
        return [sample(v, k) for v in rec[:k]]
    return rec


# ---- meta ----------------------------------------------------------------------------------------------------
def op_meta(req):
    out = {"domains": {}, "tasks": []}
    for d in DOMAINS:
        env = ENVC[d]()
        tk = env.tools
        out["domains"][d] = {
            "agent_system": SYSTEM_PROMPT.format(domain_policy=env.get_policy(), agent_instruction=AGENT_INSTRUCTION),
            "tools": [t.openai_schema for t in env.get_tools()],
            "write_tools": sorted(n for n in tk.get_tools() if tk.tool_mutates_state(n)),
            "tool_docs": {n: t.openai_schema["function"] for n, t in tk.get_tools().items()},
            "samples": {c: sample(next(iter(recs.values()))) for c, recs in dump(env).items() if recs},
        }
        for tid, task in TASKS[d].items():
            try:
                utools = env.get_user_tools(include=task.user_tools) or None
            except Exception:
                utools = None
            user = UserSimulator(llm="none", instructions=str(task.user_scenario), tools=utools)
            ec = task.evaluation_criteria
            out["tasks"].append({
                "domain": d, "id": tid, "user_system": user.system_prompt,
                "user_tools": [t.openai_schema for t in utools] if utools else None,
                "reward_basis": [str(getattr(b, "value", b)) for b in (ec.reward_basis if ec else [])],
                "gold_actions": [{"name": a.name, "arguments": a.arguments, "requestor": a.requestor}
                                 for a in ((ec.actions or []) if ec else [])],
            })
    return out


# ---- exec ----------------------------------------------------------------------------------------------------
def op_exec(req):
    res = []
    for it in req["items"]:
        env = replay(it["domain"], it["task"], it["log"])
        res.append([call(env, c) for c in it["calls"]])
    return {"results": res}


# ---- evaluate ------------------------------------------------------------------------------------------------
def to_tau2(messages):
    out = []
    for m in messages:
        tcs = [ToolCall(id=t["id"], name=t["name"], arguments=t["arguments"] if isinstance(t["arguments"], dict) else {},
                        requestor=m["role"] if m["role"] == "user" else "assistant") for t in (m.get("tool_calls") or [])]
        if m["role"] == "assistant":
            out.append(AssistantMessage(role="assistant", content=m.get("content"), tool_calls=tcs or None))
        elif m["role"] == "user":
            out.append(UserMessage(role="user", content=m.get("content"), tool_calls=tcs or None))
        elif m["role"] == "tool":
            out.append(ToolMessage(id=m["id"], role="tool", content=m.get("content"), requestor=m.get("requestor", "assistant")))
    return out


def op_evaluate(req):
    res = []
    for it in req["items"]:
        d, tid = it["domain"], it["task"]
        task = TASKS[d][tid]
        ec = task.evaluation_criteria
        env = replay(d, tid, it["log"])
        gold = new_env(d, tid)
        for a in ec.actions or []:
            try:
                gold.make_tool_call(tool_name=a.name, requestor=a.requestor, **a.arguments)
            except Exception:
                pass
        db = env.get_db_hash() == gold.get_db_hash() and env.get_user_db_hash() == gold.get_user_db_hash()
        env_ok = all(env.run_env_assertion(a, raise_assertion_error=False) for a in (ec.env_assertions or []))
        msgs = to_tau2(it["messages"])
        comm = CommunicateEvaluator.calculate_reward(task=task, full_trajectory=msgs).reward
        types = get_tool_types(env.tools)
        if env.user_tools is not None:
            types.update(get_tool_types(env.user_tools))
        act = ActionEvaluator.calculate_reward(task=task, full_trajectory=msgs, tool_types=types).reward
        parts = {"DB": float(db), "ENV_ASSERTION": float(env_ok), "COMMUNICATE": float(comm), "ACTION": float(act)}
        basis = [str(getattr(b, "value", b)) for b in ec.reward_basis]
        reward = 1.0
        for b in basis:
            if b in parts:
                reward *= parts[b]
        res.append({"reward": reward, "basis": basis, **parts})
    return {"results": res}


# ---- DB as records -------------------------------------------------------------------------------------------
def idkey(rows):
    keys = [k for k in rows[0] if k.endswith("_id") or k == "id"]
    return keys[0] if keys else None


def dump(env):
    """{collection: {record key: record}}; user-side (device) collections are prefixed 'user:'."""
    out = {}
    for side, tk in (("", env.tools), ("user:", env.user_tools)):
        if tk is None or getattr(tk, "db", None) is None:
            continue
        for coll, v in tk.db.model_dump(mode="json").items():
            if side == "":
                if isinstance(v, dict) and all(isinstance(x, dict) for x in v.values()):
                    out[coll] = v
                    continue
                if isinstance(v, list) and v and all(isinstance(x, dict) for x in v):
                    k = idkey(v)
                    out[coll] = {str(x.get(k, i)): x for i, x in enumerate(v)}
                    continue
            out[side + coll] = {coll: v}
    return out


def flat(x, p=""):
    if isinstance(x, dict) and x:
        o = {}
        for k, v in x.items():
            o.update(flat(v, f"{p}.{k}" if p else str(k)))
        return o
    if isinstance(x, list) and x:
        o = {}
        for i, v in enumerate(x):
            o.update(flat(v, f"{p}.{i}" if p else str(i)))
        return o
    return {p: x}


def set_path(rec, path, value):
    parts = path.split(".")
    cur = rec
    for s in parts[:-1]:
        cur = cur[int(s)] if isinstance(cur, list) else cur[s]
    last = parts[-1]
    if isinstance(cur, list):
        cur[int(last)] = value
    else:
        cur[last] = value


def changed_nodes(B, A):
    out = []
    for c in sorted(set(B) | set(A)):
        b, a = B.get(c, {}), A.get(c, {})
        for k in sorted(set(b) | set(a)):
            if b.get(k) != a.get(k):
                out.append((c, k))
    return out


def strings_in(x):
    if isinstance(x, dict):
        return {s for v in x.values() for s in strings_in(v)}
    if isinstance(x, list):
        return {s for v in x for s in strings_in(v)}
    return {str(x)} if x is not None else set()


def referenced(B, args):
    ss = strings_in(args)
    return [(c, k) for c in sorted(B) if not c.startswith("user:") for k in B[c] if k in ss]


def alt_value(B, coll, path, old, new, rng, top=False):
    """A plausible different value for a field: numbers shifted (as if applied twice), strings taken from the same
    field of other records. None if there is none."""
    if isinstance(new, bool) or isinstance(old, bool):
        return None if not top else (not new if isinstance(new, bool) else None)
    if isinstance(new, (int, float)):
        step = (new - old) if isinstance(old, (int, float)) and new != old else max(1, abs(new) * 0.5)
        v = new + step
        if isinstance(new, int) and not isinstance(step, float):
            v = int(v)
        else:
            v = round(v, 2)
        return v if v not in (old, new) else None
    if isinstance(new, str):
        keys = sorted(B.get(coll, {}))
        pool = set()
        for k in rng.sample(keys, min(300, len(keys))):
            v = flat(B[coll][k]).get(path)
            if isinstance(v, str) and v not in (old, new):
                pool.add(v)
        return rng.choice(sorted(pool)) if pool else None
    return None


def make_wrong(B, A, changed, args, rng, only=None):
    """Wrong-change variant of the after state: {(coll, key): record or None} overrides of B, and its kind.
    `only` restricts it to one kind (wrong_record | wrong_field | wrong_value)."""
    ss = strings_in(args)
    pre = [(c, k) for c, k in changed if not c.startswith("user:") and k in B.get(c, {}) and k in A.get(c, {})]
    prim = [x for x in pre if x[1] in ss] or pre
    options = []
    # wrong record: the intended change lands on another record of the same collection (same owner if possible)
    for c, k in prim:
        others = [k2 for k2 in B[c] if k2 != k]
        uid = B[c][k].get("user_id") or B[c][k].get("customer_id")
        same = [k2 for k2 in others if uid and (B[c][k2].get("user_id") or B[c][k2].get("customer_id")) == uid]
        if others:
            options.append(("wrong_record", (c, k, same or others)))
            break
    # wrong value: one changed leaf gets a different value
    leaves = []
    for c, k in changed:
        if c.startswith("user:") or k not in A.get(c, {}):
            continue
        fb, fa = flat(B.get(c, {}).get(k) or {}), flat(A[c][k])
        leaves += [(c, k, p, fb.get(p), fa[p]) for p in sorted(fa) if fb.get(p) != fa[p]]
    if leaves:
        options.append(("wrong_value", leaves))
    # wrong field: the intended field is left unchanged and another field of the record changes instead
    for c, k in prim:
        fields = [f for f in B[c][k] if B[c][k][f] != A[c][k].get(f)]
        others = [f for f, v in B[c][k].items() if f not in fields and isinstance(v, (str, int, float, bool))
                  and not f.endswith("_id") and f != "id"]
        if fields and others:
            options.append(("wrong_field", (c, k, fields, others)))
            break
    rng.shuffle(options)
    options = [o for o in options if only is None or o[0] == only]
    for kind, data in options:
        over = {}
        if kind == "wrong_record":
            c, k, cands = data
            k2 = rng.choice(sorted(cands))
            r2 = copy.deepcopy(B[c][k2])
            n = 0
            for f in B[c][k]:
                bv, av = B[c][k][f], A[c][k].get(f)
                if bv == av or f.endswith("_id") or f == "id":
                    continue
                if isinstance(bv, list) and isinstance(av, list) and av[:len(bv)] == bv and isinstance(r2.get(f), list):
                    r2[f] = r2[f] + copy.deepcopy(av[len(bv):])           # appended items land on the wrong record
                else:
                    r2[f] = copy.deepcopy(av)
                n += 1
            if n == 0 or r2 == B[c][k2]:
                continue
            over[(c, k)] = copy.deepcopy(B[c][k])
            over[(c, k2)] = r2
            for x in changed:                                              # other records change as intended
                if x != (c, k):
                    over[x] = copy.deepcopy(A[x[0]].get(x[1]))
            return over, kind
        if kind == "wrong_value":
            for c, k, p, old, new in rng.sample(data, len(data)):
                v = alt_value(B, c, p, old, new, rng)
                if v is None:
                    continue
                for x in changed:
                    over[x] = copy.deepcopy(A[x[0]].get(x[1]))
                set_path(over[(c, k)], p, v)
                return over, kind
            continue
        if kind == "wrong_field":
            c, k, fields, others = data
            f_rev, g = rng.choice(fields), rng.choice(sorted(others))
            v = alt_value(B, c, g, B[c][k][g], B[c][k][g], rng, top=True)
            if v is None:
                continue
            for x in changed:
                over[x] = copy.deepcopy(A[x[0]].get(x[1]))
            over[(c, k)][f_rev] = copy.deepcopy(B[c][k][f_rev])
            over[(c, k)][g] = v
            return over, kind
    return None, None


WRONG_KINDS = ("wrong_record", "wrong_field", "wrong_value")


def view(B, after, nodes, args):
    """Records shown to the verifiers: before/after of the given nodes. Large maps of sub-records (flight dates,
    product variants) keep only the entries that differ or that the call names."""
    ss = strings_in(args)
    out = {"before": {}, "after": {}}
    for c, k in nodes:
        b, a = B.get(c, {}).get(k), after.get((c, k), B.get(c, {}).get(k))
        b0, a0 = b, a
        b, a = copy.deepcopy(b), copy.deepcopy(a)
        for r, o in ((b, a0), (a, b0)):
            if not isinstance(r, dict):
                continue
            for f, v in list(r.items()):
                if isinstance(v, dict) and len(v) > 8 and all(isinstance(x, dict) for x in v.values()):
                    ov = o.get(f) if isinstance(o, dict) else None
                    r[f] = {kk: vv for kk, vv in v.items() if kk in ss or not isinstance(ov, dict) or ov.get(kk) != vv}
                    r[f]["…"] = f"{len(v) - len(r[f])} unchanged entries not shown"
        out["before"][f"{c}/{k}"] = b
        out["after"][f"{c}/{k}"] = a
    return out


def op_variants(req):
    res = []
    for it in req["items"]:
        d, tid = it["domain"], it["task"]
        env = new_env(d, tid)
        writes = set(n for n in env.tools.get_tools() if env.tools.tool_mutates_state(n))
        items = []
        for i, c in enumerate(it["log"]):
            is_w = c.get("requestor", "assistant") == "assistant" and c["name"] in writes
            if not is_w:
                call(env, c)
                continue
            B = dump(env)
            r = call(env, c)
            A = dump(env)
            changed = changed_nodes(B, A)
            if r["error"] or not changed:
                items.append({"i": i, "call": c, "output": r["content"], "error": r["error"], "skip":
                              "tool error" if r["error"] else "no state change"})
                continue
            correct = {x: A[x[0]].get(x[1]) for x in changed}
            wrongs = {k: make_wrong(B, A, changed, c.get("arguments") or {},
                                    random.Random(f"{it.get('seed', 7)}|{d}|{tid}|{i}|{k}"), only=k)[0]
                      for k in WRONG_KINDS}                       # every wrong-change kind that applies
            ref = referenced(B, c.get("arguments") or {})
            variants = {}
            for name, over in (("correct", correct), ("noop", {}), *wrongs.items()):
                if over is None:
                    continue
                nodes = list(dict.fromkeys(ref + [x for x in over if over[x] != B.get(x[0], {}).get(x[1])]))
                variants[name] = view(B, over, nodes, c.get("arguments") or {})
            items.append({"i": i, "call": c, "output": r["content"], "error": False,
                          "changed": [f"{a}/{b}" for a, b in changed], "variants": variants})
        res.append(items)
    return {"results": res}


# ---- records (notebook 4.3) ----------------------------------------------------------------------------------
def op_records(req):
    res = []
    for it in req["items"]:
        D = dump(replay(it["domain"], it["task"], it["log"]))
        res.append({n: D.get(n.split("/", 1)[0], {}).get(n.split("/", 1)[1]) for n in it["nodes"]})
    return {"results": res}


def op_named(req):
    """Records named by a call's arguments (keys found among the argument values), after replaying `log`."""
    res = []
    for it in req["items"]:
        D = dump(replay(it["domain"], it["task"], it["log"]))
        res.append({f"{c}/{k}": D[c][k] for c, k in referenced(D, it["args"])})
    return {"results": res}


OPS = {"meta": op_meta, "exec": op_exec, "evaluate": op_evaluate, "variants": op_variants, "records": op_records,
       "named": op_named}

if __name__ == "__main__":
    for line in sys.stdin:
        if not line.strip():
            continue
        try:
            req = json.loads(line)
            out = OPS[req["op"]](req)
        except Exception as e:                                        # report, keep serving
            import traceback
            out = {"error": f"{type(e).__name__}: {e}", "trace": traceback.format_exc()[-2000:]}
        _OUT.write(json.dumps(out, default=str) + "\n")
        _OUT.flush()

Overwriting data/benchmarks/tau2/tau2_env_server.py


In [4]:
class Tau2Server:
    def __init__(self):
        self.p = subprocess.Popen([PYV, "-I", f"{BASE}/tau2_env_server.py"], stdin=subprocess.PIPE,
                                  stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
    def ask(self, req):
        self.p.stdin.write(json.dumps(req) + "\n"); self.p.stdin.flush()
        r = json.loads(self.p.stdout.readline())
        if "error" in r:
            raise RuntimeError(r["error"] + "\n" + r.get("trace", ""))
        return r

srv = Tau2Server()
META = srv.ask({"op": "meta"})
DOM = META["domains"]
TASK = {f"{t['domain']}|{t['id']}": t for t in META["tasks"]}
print("tasks per domain (base split):", dict(Counter(t["domain"] for t in META["tasks"])), "| total", len(TASK))
for d, m in DOM.items():
    print(f"  {d:<8} agent prompt {len(m['agent_system']):>6,} chars, {len(m['tools'])} tools, write tools: {', '.join(m['write_tools'])}")
BY_DOM = defaultdict(list)
for k in TASK:
    BY_DOM[k.split("|")[0]].append(k)
KEYS = [k for d, ks in BY_DOM.items() for k in (ks[:TEST_TASKS[d]] if TEST else ks)]
print(f"{len(KEYS)} tasks in this run:", dict(Counter(k.split('|')[0] for k in KEYS)))

tasks per domain (base split): {'airline': 50, 'retail': 114, 'telecom': 114} | total 278
  airline  agent prompt  8,033 chars, 14 tools, write tools: book_reservation, cancel_reservation, send_certificate, update_reservation_baggages, update_reservation_flights, update_reservation_passengers
  retail   agent prompt  7,056 chars, 16 tools, write tools: cancel_pending_order, exchange_delivered_order_items, modify_pending_order_address, modify_pending_order_items, modify_pending_order_payment, modify_user_address, return_delivered_order_items
  telecom  agent prompt 23,675 chars, 13 tools, write tools: disable_roaming, enable_roaming, refuel_data, resume_line, send_payment_request, suspend_line
278 tasks in this run: {'airline': 50, 'retail': 114, 'telecom': 114}


## Recording: agent and user simulator, one LLM round per step

The conversation loop follows τ²-bench's orchestrator: the agent opens with its fixed greeting; the user simulator and the
agent take turns; a turn with tool calls is executed by the environment and the same party continues; the user ends with
`###STOP###` (or `###TRANSFER###`, `###OUT-OF-SCOPE###`). The agent sees its own tool calls and results, the user simulator
only its own (roles flipped, as in τ²-bench). Each round is one `llm.run` over all live conversations.

In [5]:
STOP_TOKENS = ("###STOP###", "###TRANSFER###", "###OUT-OF-SCOPE###")
GREETING = "Hi! How can I help you today?"          # tau2-bench's fixed first agent message
MAX_STEPS, MAX_ERRORS = 100, 10                      # tau2-bench defaults (messages incl. tool results; tool errors)

def oa_calls(tcs):
    return [{"id": t["id"], "type": "function", "function": {"name": t["name"], "arguments": t["raw"]}} for t in tcs]

def agent_view(conv, events=None):
    """What the agent LLM sees: its policy, the dialogue, and its own tool calls and results (not the user's)."""
    msgs = [{"role": "system", "content": DOM[conv["domain"]]["agent_system"]}]
    for e in (conv["events"] if events is None else events):
        if e["role"] == "assistant":
            m = {"role": "assistant", "content": e["content"]}
            if e["tool_calls"]:
                m["tool_calls"] = oa_calls(e["tool_calls"])
            msgs.append(m)
        elif e["role"] == "user" and not e["tool_calls"]:
            msgs.append({"role": "user", "content": e["content"]})
        elif e["role"] == "tool" and e["requestor"] == "assistant":
            msgs.append({"role": "tool", "tool_call_id": e["id"], "content": e["content"]})
        elif e["role"] == "note":                     # notebook 4.3: resume notes for the agent
            msgs.append({"role": "user", "content": e["content"]})
    return msgs

def user_view(conv):
    """What the user simulator sees (roles flipped, as in tau2-bench): agent messages as 'user', its own as
    'assistant', its own tool calls and results; the agent's tool calls are hidden."""
    msgs = [{"role": "system", "content": TASK[conv["key"]]["user_system"]}]
    for e in conv["events"]:
        if e["role"] == "assistant" and not e["tool_calls"] and e["content"]:
            msgs.append({"role": "user", "content": e["content"]})
        elif e["role"] == "user":
            m = {"role": "assistant", "content": e["content"]}
            if e["tool_calls"]:
                m["tool_calls"] = oa_calls(e["tool_calls"])
            msgs.append(m)
        elif e["role"] == "tool" and e["requestor"] == "user":
            msgs.append({"role": "tool", "tool_call_id": e["id"], "content": e["content"]})
    return msgs

def new_conv(key, cid=None):
    d, t = key.split("|", 1)
    return {"id": cid or key, "key": key, "domain": d, "task": t, "turn": "user", "done": False, "term": None,
            "steps": 1, "errors": 0, "llm_calls": 0, "tokens": {"agent_in": 0, "agent_out": 0, "user_in": 0, "user_out": 0},
            "events": [{"role": "assistant", "content": GREETING, "tool_calls": []}], "log": []}

def parse_calls(r):
    out = []
    for t in r.get("tool_calls") or []:
        try:
            args = json.loads(t["arguments"]) if t["arguments"] else {}
            args = args if isinstance(args, dict) else None
        except json.JSONDecodeError:
            args = None
        out.append({"id": t["id"], "name": t["name"], "raw": t["arguments"], "arguments": args})
    return out

def agent_spoke(c):
    return any(e["role"] == "assistant" and not e["tool_calls"] and e["content"] for e in c["events"][1:])

def body_for(c):
    if c["turn"] == "agent":
        b = LLM.body(agent_view(c), max_tokens=1500, tools=DOM[c["domain"]]["tools"])
    else:   # the user's device tools (telecom) only after the agent has answered once (see the protocol changes)
        b = LLM.body(user_view(c), max_tokens=1000, tools=TASK[c["key"]]["user_tools"] if agent_spoke(c) else None)
    b["seed"] = c.get("seed", 7)                    # recording pass (Notebook 4.4); 7 = smash_batch's default
    return b

# Each lock-step round is one Batch job, and the Batch queue took hours per round (2026-11-09/10). STEP_TIER = "standard"
# sends the rounds synchronously (standard tier, 2x price; same model and settings, tier only), as notebooks 1.2b and 2.3.
# 2026-11-10: switched at step 31 of the recording pass (steps 1-30 ran via Batch). Other stages (verifiers, judges) still use the Batch API.
STEP_TIER = "standard"

def simulate(convs, stage, max_steps=MAX_STEPS, step0=0, on_step=None):
    """Run all conversations in lockstep: every round is ONE llm.run over all live conversations (each needs either
    an agent or a user-simulator turn), so in batch mode each round is one Batch job. Tool calls are executed by
    the tau2 server between rounds."""
    n = step0
    while True:
        live = [c for c in convs if not c["done"]]
        if not live:
            break
        n += 1
        mode0 = llm.mode
        if MODE == "batch" and STEP_TIER == "standard":
            llm.mode = "test"                        # synchronous standard tier for the lock-step rounds only
        try:
            res = llm.run(f"{stage}_step{n}", {c["id"]: body_for(c) for c in live})
        finally:
            llm.mode = mode0
        todo = []
        for c in live:
            r, role = res[c["id"]], c["turn"]
            c["llm_calls"] += 1
            if role == "agent" and "first_agent_in" not in c:
                c["first_agent_in"] = r.get("in", 0)           # prompt size of the agent's first turn (Notebook 4.3)
            c["tokens"][f"{role}_in"] += r.get("in", 0); c["tokens"][f"{role}_out"] += r.get("out", 0)
            if "error" in r:
                c["done"], c["term"] = True, f"{role}_llm_error"; continue
            tcs = parse_calls(r)
            c["events"].append({"role": "assistant" if role == "agent" else "user", "content": r["text"] or None,
                                "tool_calls": tcs})
            c["steps"] += 1
            if tcs:
                todo.append(c)
            elif not r["text"]:
                c["done"], c["term"] = True, f"{role}_error"
            elif role == "user" and any(s in r["text"] for s in STOP_TOKENS):
                c["done"], c["term"] = True, "user_stop"
            else:
                c["turn"] = "user" if role == "agent" else "agent"
        if todo:
            items = [{"domain": c["domain"], "task": c["task"], "log": c["log"],
                      "calls": [{"id": t["id"], "name": t["name"], "arguments": t["arguments"],
                                 "requestor": "assistant" if c["turn"] == "agent" else "user"}
                                for t in c["events"][-1]["tool_calls"] if t["arguments"] is not None]} for c in todo]
            out = srv.ask({"op": "exec", "items": items})["results"]
            for c, it, rs in zip(todo, items, out):
                rq = "assistant" if c["turn"] == "agent" else "user"
                rs = iter(rs)
                for t in c["events"][-1]["tool_calls"]:
                    if t["arguments"] is None:
                        o = {"content": "Error: the tool call arguments are not valid JSON.", "error": True}
                    else:
                        o = next(rs)
                        c["log"].append({"id": t["id"], "name": t["name"], "arguments": t["arguments"], "requestor": rq,
                                         "error": o["error"], "event": len(c["events"]) - 1})
                    c["events"].append({"role": "tool", "id": t["id"], "name": t["name"], "content": o["content"],
                                        "requestor": rq, "error": o["error"]})
                    c["errors"] += o["error"]
                c["steps"] += 1
        for c in live:
            if not c["done"] and c["steps"] >= max_steps:
                c["done"], c["term"] = True, "max_steps"
            elif not c["done"] and c["errors"] >= MAX_ERRORS:
                c["done"], c["term"] = True, "too_many_errors"
        if on_step:
            on_step(n, live)
    return n

def evaluate(convs, extra_events=None):
    """tau2-bench's state-based reward on the environment after every executed tool call (c['log']); conversations
    that did not end with a user stop get 0, as in tau2-bench."""
    items = []
    for c in convs:
        msgs = []
        for e in (extra_events or {}).get(c["id"], []) + c["events"]:
            if e["role"] in ("assistant", "user"):
                msgs.append({"role": e["role"], "content": e["content"],
                             "tool_calls": [{"id": t["id"], "name": t["name"], "arguments": t["arguments"]}
                                            for t in e["tool_calls"] if t["arguments"] is not None]})
            elif e["role"] == "tool":
                msgs.append({"role": "tool", "id": e["id"], "content": e["content"], "requestor": e["requestor"]})
        items.append({"domain": c["domain"], "task": c["task"], "log": c["log"], "messages": msgs})
    out = srv.ask({"op": "evaluate", "items": items})["results"]
    for c, r in zip(convs, out):
        c["eval"] = r
        c["reward"] = r["reward"] if c["term"] == "user_stop" else 0.0

In [6]:
SEEDS = (7, 8, 9)                    # three recording passes; pass 1 (seed 7) = Notebook 4.3's reference runs
t0 = time.time()
ALL = [dict(new_conv(k, cid=k if s == 7 else f"{k}#s{s}"), seed=s) for s in SEEDS for k in KEYS]
rounds = simulate(ALL, "nb28_rec", on_step=lambda n, live: print(f"round {n}: {len(live)} live") if n % 10 == 0 else None)
evaluate(ALL)
REC = [c for c in ALL if c["seed"] == 7]
print(f"{len(ALL)} conversations ({len(SEEDS)} passes x {len(KEYS)} tasks) in {rounds} rounds, {time.time() - t0:.0f} s")
same = {s: np.mean([[e["content"] for e in c["events"]] == [e["content"] for e in r["events"]]
                    for c, r in zip([c for c in ALL if c["seed"] == s], REC)]) for s in SEEDS[1:]}
print("reward per pass:", {s: f"{np.mean([c['reward'] for c in ALL if c['seed'] == s]):.0%}" for s in SEEDS},
      "| conversations identical to pass 1:", {s: f"{v:.0%}" for s, v in same.items()})
print("pass 1 only:")
print(f"{'domain':<9}{'n':>4}{'reward':>8}{'LLM calls':>11}{'steps':>7}{'agent in tok':>14}{'user in tok':>13}  end reasons")
for d in DOM:
    cs = [c for c in REC if c["domain"] == d]
    if cs:
        print(f"{d:<9}{len(cs):>4}{np.mean([c['reward'] for c in cs]):>8.0%}{np.mean([c['llm_calls'] for c in cs]):>11.1f}"
              f"{np.mean([c['steps'] for c in cs]):>7.1f}{np.mean([c['tokens']['agent_in'] for c in cs]):>14,.0f}"
              f"{np.mean([c['tokens']['user_in'] for c in cs]):>13,.0f}  {dict(Counter(c['term'] for c in cs))}")
json.dump({"commit": TAU2_COMMIT, "mode": MODE, "convs": REC, "convs_extra": [c for c in ALL if c["seed"] != 7]},
          open(f"{BASE}/nb28_recordings" + ("_test" if TEST else "") + ".json", "w"))

round 10: 762 live


round 20: 316 live


round 30: 73 live
[nb28_rec_step31] 59 new requests -> test (0 cached)


[nb28_rec_step32] 49 new requests -> test (0 cached)


[nb28_rec_step33] 41 new requests -> test (0 cached)


[nb28_rec_step34] 38 new requests -> test (0 cached)


... (28 outputs left out)

[nb28_rec_step63] 1 new requests -> test (0 cached)


[nb28_rec_step64] 1 new requests -> test (0 cached)


[nb28_rec_step65] 1 new requests -> test (0 cached)


834 conversations (3 passes x 278 tasks) in 65 rounds, 294 s
reward per pass: {7: '50%', 8: '46%', 9: '51%'} | conversations identical to pass 1: {8: '1%', 9: '1%'}
pass 1 only:
domain      n  reward  LLM calls  steps  agent in tok  user in tok  end reasons
airline    50     52%       13.6   20.6        51,318        2,988  {'user_stop': 50}
retail    114     72%       19.0   29.0        75,522        4,059  {'user_stop': 113, 'max_steps': 1}
telecom   114     26%       20.0   31.9        88,530       21,166  {'user_stop': 113, 'max_steps': 1}


## The variants of every write action

The server replays each recording; at every successful agent write it takes the database before and after the call and
builds the no-op and the three wrong-change variants. Actions repeated across passes (same task, tool, arguments and
before-state) are kept once. A variant is a set of records (the records the call names, plus those that
changed in that variant) before and after.

In [7]:
VAR = srv.ask({"op": "variants", "items": [{"domain": c["domain"], "task": c["task"], "log": c["log"], "seed": 7}
                                           for c in ALL]})["results"]
ITEMS, skipped, seen, dups = [], Counter(), set(), Counter()
for c, vs in zip(ALL, VAR):
    for v in vs:
        if "variants" not in v:
            skipped[v["skip"]] += 1
            continue
        key = (c["key"], v["call"]["name"], json.dumps(v["call"]["arguments"], sort_keys=True),
               json.dumps(v["variants"]["correct"]["before"], sort_keys=True))
        if key in seen:
            dups[c["seed"]] += 1
            continue
        seen.add(key)
        ITEMS.append({"conv": c, **v})
print(f"agent write calls in {len(SEEDS)} passes: {len(ITEMS) + sum(dups.values()) + sum(skipped.values())}; "
      f"skipped {dict(skipped)}; duplicates of an earlier pass removed {dict(dups)}; distinct eligible {len(ITEMS)}")
print("distinct actions per pass:", dict(Counter(it["conv"]["seed"] for it in ITEMS)))
print("eligible per domain:", dict(Counter(it["conv"]["domain"] for it in ITEMS)),
      "| per tool:", dict(Counter(it["call"]["name"] for it in ITEMS)))
print("actions with each wrong-change kind:", dict(Counter(k for it in ITEMS for k in it["variants"] if k.startswith("wrong"))))
print(f"distinct write actions per task (3 passes): {len(ITEMS) / len(KEYS):.2f}")

agent write calls in 3 passes: 843; skipped {'tool error': 13, 'no state change': 23}; duplicates of an earlier pass removed {8: 187, 9: 222}; distinct eligible 398
distinct actions per pass: {7: 266, 8: 85, 9: 47}
eligible per domain: {'airline': 55, 'retail': 216, 'telecom': 127} | per tool: {'cancel_reservation': 21, 'update_reservation_flights': 19, 'update_reservation_baggages': 8, 'book_reservation': 5, 'update_reservation_passengers': 2, 'exchange_delivered_order_items': 46, 'return_delivered_order_items': 54, 'modify_pending_order_items': 45, 'cancel_pending_order': 28, 'modify_user_address': 14, 'modify_pending_order_address': 28, 'modify_pending_order_payment': 1, 'refuel_data': 47, 'enable_roaming': 42, 'send_payment_request': 22, 'resume_line': 15, 'suspend_line': 1}
actions with each wrong-change kind: {'wrong_record': 344, 'wrong_field': 225, 'wrong_value': 341}
distinct write actions per task (3 passes): 1.43


## Verifier (3): the database state as a graph, the expected change as a pattern

`state_graph` turns the records into a property graph; `graph_diff` compares the graphs before and after; `check`
matches a tool's pattern against the diff. A pattern binds node variables from the call's arguments
(`reservations/{reservation_id}`, `new:reservations` for a created record, `flights/{flights[*].flight_number}` for one
node per list element), lists the expected changes (`eq`, `eq_list`, `contains_all`, `delta`, `changed`, `appended`,
`new_value` on a property path, `*` wildcards allowed), and a **frame**: the only nodes allowed to change. The action
passes if every expected change is present and nothing outside the frame changed.

In [8]:
def flat(x, p=""):
    if isinstance(x, dict) and x:
        o = {}
        for k, v in x.items():
            o.update(flat(v, f"{p}.{k}" if p else str(k)))
        return o
    if isinstance(x, list) and x:
        o = {}
        for i, v in enumerate(x):
            o.update(flat(v, f"{p}.{i}" if p else str(i)))
        return o
    return {p: x}

def state_graph(records):
    """DB state as a property graph: one node per record ('collection/key'), its fields (flattened paths) as
    properties, and an edge wherever a field holds the key of another record in view (reservation -> user)."""
    G = nx.DiGraph()
    keys = {n.split("/", 1)[1]: n for n, r in records.items() if r is not None}
    for n, r in records.items():
        if r is None:
            continue
        props = flat(r)
        G.add_node(n, label=n.split("/", 1)[0], props=props)
        for path, v in props.items():
            if isinstance(v, str) and v in keys and keys[v] != n:
                G.add_edge(n, keys[v], rel=path)
    return G

def graph_diff(Gb, Ga):
    """Node-level diff of two state graphs: added / removed nodes, changed properties, added / removed edges."""
    out = {}
    for n in sorted(set(Gb) | set(Ga)):
        pb = Gb.nodes[n]["props"] if n in Gb else None
        pa = Ga.nodes[n]["props"] if n in Ga else None
        if pb == pa:
            continue
        b, a = pb or {}, pa or {}
        out[n] = {"added": pb is None, "removed": pa is None,
                  "changes": {p: (b.get(p), a.get(p)) for p in sorted(set(b) | set(a)) if b.get(p) != a.get(p)}}
    edges = {"added": sorted(set(Ga.edges) - set(Gb.edges)), "removed": sorted(set(Gb.edges) - set(Ga.edges))}
    return out, edges

# ---- expected-change patterns: matching against the diff ---------------------------------------------------
def getp(x, path):
    """Argument lookup: 'reservation_id', 'flights[*].date' (a list), 'passengers[*].first_name'."""
    cur = [x]
    for tok in path.split("."):
        star = tok.endswith("[*]")
        tok = tok[:-3] if star else tok
        nxt = []
        for c in cur:
            v = c.get(tok) if isinstance(c, dict) else None
            if star and isinstance(v, list):
                nxt += v
            elif v is not None:
                nxt.append(v)
        cur = nxt
    return cur if "[*]" in path else (cur[0] if cur else None)

def resolve(tmpl, args):
    """'{x}' -> the argument value (a list for '{xs[*].f}'); other strings are literals."""
    if not isinstance(tmpl, str):
        return tmpl
    m = re.fullmatch(r"\{([^{}]+)\}", tmpl.strip())
    return getp(args, m.group(1)) if m else tmpl

def expand(tmpl, args):
    """Node template -> concrete node ids; a list-valued placeholder gives one id per element."""
    out = [tmpl]
    for ph in re.findall(r"\{([^{}]+)\}", tmpl):
        v = getp(args, ph)
        out = [o.replace("{" + ph + "}", str(x)) for o in out for x in as_list(v)]
    return out

def same(a, b):
    if isinstance(a, bool) or isinstance(b, bool) or str(a).lower() in ("true", "false") or str(b).lower() in ("true", "false"):
        return str(a).lower() == str(b).lower()
    try:
        return abs(float(a) - float(b)) < 1e-6
    except (TypeError, ValueError):
        return str(a) == str(b)

def under(props, path):
    """Leaves at `path` or below it; `path` may contain * wildcards (fnmatch)."""
    if not path or path == "*":
        return dict(props)
    if any(ch in path for ch in "*?["):
        return {p: v for p, v in props.items() if fnmatch.fnmatchcase(p, path) or fnmatch.fnmatchcase(p, path + ".*")}
    return {p: v for p, v in props.items() if p == path or p.startswith(path + ".")}

def as_list(v):
    return v if isinstance(v, list) else [v]

def clause_ok(cl, pb, pa, args):
    op, path = cl["op"], cl.get("path", "")
    val = resolve(cl.get("value"), args)
    b, a = under(pb, path), under(pa, path)
    if op == "eq":
        return bool(a) and all(same(v, val) for v in a.values())
    if op == "eq_list":
        return sorted(map(str, a.values())) == sorted(map(str, as_list(val)))
    if op == "contains_all":
        have = [str(v) for v in a.values()]
        return all(any(same(h, x) for h in have) for x in as_list(val))
    if op == "delta":
        try:
            return all(abs(float(a.get(p, 0) or 0) - float(b.get(p, 0) or 0) - float(val)) < 1e-6 for p in (a or {path: 0}))
        except (TypeError, ValueError):
            return False
    if op == "changed":
        return a != b
    if op == "appended":
        idx = lambda d: {p[len(path) + 1:].split(".")[0] for p in d if p.startswith(path + ".")}
        return len(idx(a)) > len(idx(b))
    if op == "new_value":
        return any(same(v, val) and not (p in b and same(b[p], v)) for p, v in a.items())
    raise ValueError(f"unknown op {op}")

def node_ids(tmpl, binds, args, diff, universe):
    """-> (candidate node ids, all_must_hold). 'new:coll' = the record added in that collection; a template with a
    list-valued placeholder names several records (the clause must hold for each); a * wildcard matches any."""
    t = binds.get(tmpl, tmpl)
    if t.startswith("new:"):
        return [n for n, d in diff.items() if d["added"] and n.split("/", 1)[0] == t[4:]], False
    ids = expand(t, args)
    if any(ch in t for ch in "*?"):
        return [n for n in universe if any(fnmatch.fnmatchcase(n, i) for i in ids)], False
    return ids, len(ids) > 1

def check(pattern, call_args, view):
    """True if the state change in `view` matches the tool's expected-change pattern; reasons otherwise."""
    Gb, Ga = state_graph(view["before"]), state_graph(view["after"])
    diff, _ = graph_diff(Gb, Ga)
    universe = sorted(set(Gb) | set(Ga))
    binds = {b["var"]: b["node"] for b in pattern.get("bind", [])}
    why = []
    for cl in pattern.get("expect", []):
        cands, every = node_ids(cl["node"], binds, call_args, diff, universe)
        props = lambda G, n: G.nodes[n]["props"] if n in G else {}
        hits = [clause_ok(cl, props(Gb, n), props(Ga, n), call_args) for n in cands]
        if not (hits and (all(hits) if every else any(hits))):
            why.append(f"expected {cl['op']} {cl['node']}.{cl.get('path', '')} {cl.get('value', '')}".strip())
    allowed = [x for f in pattern.get("frame", []) for x in
               ([binds.get(f, f)[4:] + "/*"] if binds.get(f, f).startswith("new:") else expand(binds.get(f, f), call_args))]
    for n in diff:
        if not any(fnmatch.fnmatchcase(n, t) for t in allowed):
            why.append(f"unexpected change in {n}")
    if not diff:
        why.append("no state change")
    return not why, why

### The 19 hand-written patterns (one per write tool)

Written from τ²-bench's tool code and docstrings. Effort: 19 patterns with 53 expected-change clauses, written in one pass over the tool code and checked once on replays of the gold actions (see the protocol).

In [9]:
C = lambda node, op, path="", value=None: {"node": node, "op": op, "path": path, **({"value": value} if value is not None else {})}
B = lambda var, node: {"var": var, "node": node}
ADDR = ("address1", "address2", "city", "state", "country", "zip")

PATTERNS = {
    # ---- airline ----
    "book_reservation": {"bind": [B("R", "new:reservations"), B("U", "users/{user_id}"),
                                  B("F", "flights/{flights[*].flight_number}")], "expect": [
        C("R", "eq", "user_id", "{user_id}"), C("R", "eq", "cabin", "{cabin}"), C("R", "eq", "origin", "{origin}"),
        C("R", "eq", "destination", "{destination}"), C("R", "eq", "insurance", "{insurance}"),
        C("R", "eq", "total_baggages", "{total_baggages}"), C("R", "eq", "nonfree_baggages", "{nonfree_baggages}"),
        C("R", "contains_all", "flights.*.flight_number", "{flights[*].flight_number}"),
        C("R", "contains_all", "flights.*.date", "{flights[*].date}"),
        C("R", "contains_all", "passengers.*.first_name", "{passengers[*].first_name}"),
        C("U", "appended", "reservations"), C("F", "changed", "dates")], "frame": ["R", "U", "F"]},
    "cancel_reservation": {"bind": [B("R", "reservations/{reservation_id}")],
                           "expect": [C("R", "eq", "status", "cancelled")], "frame": ["R"]},
    "send_certificate": {"bind": [B("U", "users/{user_id}")],
                         "expect": [C("U", "new_value", "payment_methods.*.amount", "{amount}")], "frame": ["U"]},
    "update_reservation_baggages": {"bind": [B("R", "reservations/{reservation_id}")], "expect": [
        C("R", "eq", "total_baggages", "{total_baggages}"), C("R", "eq", "nonfree_baggages", "{nonfree_baggages}")],
        "frame": ["R", "users/*"]},
    "update_reservation_flights": {"bind": [B("R", "reservations/{reservation_id}")], "expect": [
        C("R", "eq", "cabin", "{cabin}"), C("R", "eq_list", "flights.*.flight_number", "{flights[*].flight_number}"),
        C("R", "eq_list", "flights.*.date", "{flights[*].date}")], "frame": ["R", "users/*"]},
    "update_reservation_passengers": {"bind": [B("R", "reservations/{reservation_id}")], "expect": [
        C("R", "eq_list", "passengers.*.first_name", "{passengers[*].first_name}"),
        C("R", "eq_list", "passengers.*.last_name", "{passengers[*].last_name}"),
        C("R", "eq_list", "passengers.*.dob", "{passengers[*].dob}")], "frame": ["R"]},
    # ---- retail ----
    "cancel_pending_order": {"bind": [B("O", "orders/{order_id}")], "expect": [
        C("O", "eq", "status", "cancelled"), C("O", "eq", "cancel_reason", "{reason}")], "frame": ["O", "users/*"]},
    "exchange_delivered_order_items": {"bind": [B("O", "orders/{order_id}")], "expect": [
        C("O", "eq", "status", "exchange requested"), C("O", "eq_list", "exchange_items", "{item_ids}"),
        C("O", "eq_list", "exchange_new_items", "{new_item_ids}"),
        C("O", "eq", "exchange_payment_method_id", "{payment_method_id}")], "frame": ["O"]},
    "modify_pending_order_address": {"bind": [B("O", "orders/{order_id}")],
                                     "expect": [C("O", "eq", f"address.{f}", "{%s}" % f) for f in ADDR], "frame": ["O"]},
    "modify_pending_order_items": {"bind": [B("O", "orders/{order_id}")], "expect": [
        C("O", "eq", "status", "pending (item modified)"), C("O", "contains_all", "items.*.item_id", "{new_item_ids}"),
        C("O", "appended", "payment_history")], "frame": ["O", "users/*"]},
    "modify_pending_order_payment": {"bind": [B("O", "orders/{order_id}")], "expect": [
        C("O", "new_value", "payment_history.*.payment_method_id", "{payment_method_id}")], "frame": ["O", "users/*"]},
    "modify_user_address": {"bind": [B("U", "users/{user_id}")],
                            "expect": [C("U", "eq", f"address.{f}", "{%s}" % f) for f in ADDR], "frame": ["U"]},
    "return_delivered_order_items": {"bind": [B("O", "orders/{order_id}")], "expect": [
        C("O", "eq", "status", "return requested"), C("O", "eq_list", "return_items", "{item_ids}"),
        C("O", "eq", "return_payment_method_id", "{payment_method_id}")], "frame": ["O"]},
    # ---- telecom ----
    "enable_roaming": {"bind": [B("L", "lines/{line_id}")], "expect": [C("L", "eq", "roaming_enabled", "true")],
                       "frame": ["L", "user:*"]},
    "disable_roaming": {"bind": [B("L", "lines/{line_id}")], "expect": [C("L", "eq", "roaming_enabled", "false")],
                        "frame": ["L", "user:*"]},
    "refuel_data": {"bind": [B("L", "lines/{line_id}")], "expect": [C("L", "delta", "data_refueling_gb", "{gb_amount}")],
                    "frame": ["L", "bills/*", "customers/{customer_id}", "user:*"]},
    "resume_line": {"bind": [B("L", "lines/{line_id}")], "expect": [C("L", "eq", "status", "Active")],
                    "frame": ["L", "user:*"]},
    "suspend_line": {"bind": [B("L", "lines/{line_id}")], "expect": [C("L", "eq", "status", "Suspended")],
                     "frame": ["L", "user:*"]},
    "send_payment_request": {"bind": [B("X", "bills/{bill_id}")], "expect": [C("X", "eq", "status", "Awaiting Payment")],
                             "frame": ["X", "user:*"]},
}

n_cl = sum(len(p["expect"]) for p in PATTERNS.values())
write_tools = sorted({t for d in DOM.values() for t in d["write_tools"]})
print(f"{len(PATTERNS)} hand-written patterns, {n_cl} expected-change clauses; write tools without a pattern:",
      sorted(set(write_tools) - set(PATTERNS)) or "none")
it = next((x for x in ITEMS if "wrong_record" in x["variants"]), ITEMS[0] if ITEMS else None)
if it:
    print(f"\nExample: {it['call']['name']}({json.dumps(it['call']['arguments'])[:200]})")
    for name, view in it["variants"].items():
        ok, why = check(PATTERNS[it["call"]["name"]], it["call"]["arguments"], view)
        dif, _ = graph_diff(state_graph(view["before"]), state_graph(view["after"]))
        print(f"  {name:<8} changed nodes {sorted(dif)} -> {'PASS' if ok else 'FAIL: ' + '; '.join(why)}")

19 hand-written patterns, 53 expected-change clauses; write tools without a pattern: none

Example: cancel_reservation({"reservation_id": "59XX6W"})
  correct  changed nodes ['reservations/59XX6W'] -> PASS
  noop     changed nodes [] -> FAIL: expected eq R.status cancelled; no state change
  wrong_record changed nodes ['reservations/9MRJD4'] -> FAIL: expected eq R.status cancelled; unexpected change in reservations/9MRJD4
  wrong_field changed nodes ['reservations/59XX6W'] -> FAIL: expected eq R.status cancelled
  wrong_value changed nodes ['reservations/59XX6W'] -> PASS


### Variant (3b): patterns written by the LLM

One call per write tool: the pattern language (the same documentation as above), the tool's description and parameters,
and one sample record of each collection of its domain. The LLM's pattern is used as written.

In [10]:
class PBind(BaseModel):
    var: str
    node: str
class PClause(BaseModel):
    node: str
    op: str
    path: str
    value: str
class PatternOut(BaseModel):
    bind: list[PBind]
    expect: list[PClause]
    frame: list[str]

PATTERN_DOC = """You write the expected-change pattern of a database write tool. The database is a graph: each record is a node
named 'collection/key' (for example 'orders/#W123'); its fields are properties addressed by dotted paths ('status',
'address.city', 'items.0.item_id'; list elements by index; '*' matches any index or key). Device-side records in the
telecom domain are named 'user:device/device' and 'user:surroundings/surroundings'.

A pattern has:
- bind: node variables. 'node' is a template filled from the call's arguments: 'orders/{order_id}'; a list argument
  names one node per element: 'flights/{flights[*].flight_number}'; 'new:reservations' binds the record the call creates.
- expect: clauses {node (a variable or a template), op, path, value}. 'value' may be a literal or an argument template
  such as '{status}' or '{item_ids}' (use '' when the op takes no value). Ops:
  eq (every property at path equals value), eq_list (the list at path equals the list value, any order),
  contains_all (the properties at path contain every element of value), delta (after - before equals value),
  changed (something at path changed), appended (the list at path grew), new_value (a property at path newly has value).
- frame: the nodes allowed to change (variables, templates, or wildcards like 'users/*'); any other change is a failure.
The action is judged successful only if every expect clause holds and nothing outside the frame changed.

Example for a hypothetical tool close_ticket(ticket_id, resolution) that sets the ticket's status to 'closed' and its
resolution, and appends to the ticket's history:
{"bind": [{"var": "T", "node": "tickets/{ticket_id}"}],
 "expect": [{"node": "T", "op": "eq", "path": "status", "value": "closed"},
            {"node": "T", "op": "eq", "path": "resolution", "value": "{resolution}"},
            {"node": "T", "op": "appended", "path": "history", "value": ""}],
 "frame": ["T"]}"""

def pattern_prompt(d, tool):
    samples = json.dumps(DOM[d]["samples"], indent=1)[:12000]
    return (f"{PATTERN_DOC}\n\nDomain: {d}. One sample record per collection:\n{samples}\n\n"
            f"Tool:\n{json.dumps(DOM[d]['tool_docs'][tool], indent=1)}\n\nWrite the pattern for this tool.")

tool_dom = {t: d for d, m in DOM.items() for t in m["write_tools"]}
pres = llm.run("nb28_pattern_llm", {t: LLM.body([{"role": "user", "content": pattern_prompt(d, t)}], max_tokens=1500,
                                                 schema=PatternOut) for t, d in tool_dom.items()}, parse=PatternOut)
LLM_PATTERNS = {t: r["parsed"] for t, r in pres.items() if r["parsed"]}
print(f"LLM-written patterns: {len(LLM_PATTERNS)} of {len(tool_dom)} parsed")
for t in list(LLM_PATTERNS)[:2]:
    print(f"  {t}: {json.dumps(LLM_PATTERNS[t])[:400]}")

[nb28_pattern_llm] 19 new requests -> batch (0 cached)


[nb28_pattern_llm] uploaded input file 1/1 (19 requests, 0 MB)


[nb28_pattern_llm] started batch batch_6af2f96aa4a88190b3239facc258fd85


[nb28_pattern_llm] 1 min: 1 running, 0 queued, 0 results cached


[nb28_pattern_llm] batch batch_6af2f96aa4a88190b3239facc258fd85 completed: 19 ok, 0 failed


[nb28_pattern_llm] done in 2.6 min


LLM-written patterns: 19 of 19 parsed
  book_reservation: {"bind": [{"var": "R", "node": "new:reservations"}, {"var": "U", "node": "users/{user_id}"}], "expect": [{"node": "R", "op": "eq", "path": "user_id", "value": "{user_id}"}, {"node": "R", "op": "eq", "path": "origin", "value": "{origin}"}, {"node": "R", "op": "eq", "path": "destination", "value": "{destination}"}, {"node": "R", "op": "eq", "path": "flight_type", "value": "{flight_type}"}, {"node": 
  cancel_reservation: {"bind": [{"var": "R", "node": "reservations/{reservation_id}"}], "expect": [{"node": "R", "op": "eq", "path": "status", "value": "cancelled"}], "frame": ["R"]}


## Verifiers (1) and (2): the LLM asked whether the action succeeded

Same system prompt and the same conversation, call, tool description and output for both; (2) also sees the records
before and after. The prompt states the threat (a tool may report success although nothing changed, or change something
else), so the LLM is not left to trust the output by default. One structured call per variant: a one-sentence reason,
then the verdict.

In [11]:
class Verdict(BaseModel):
    reason: str = Field(description="one sentence")
    succeeded: bool

VERIFY_SYS = ("You verify the actions of a customer-service agent. An action is a call to a tool that writes to the "
              "database. Decide whether it succeeded: the database now reflects the change the call requested (on the "
              "record it named, in the fields and with the values it asked for), and nothing else was changed. Tool "
              "outputs are not always reliable: a tool may report success although nothing changed, or although it "
              "changed something else. Give a one-sentence reason, then the verdict.")
CAP = 1500                                                     # characters per earlier tool result in the transcript

def transcript(conv, ev_idx):
    lines = []
    for e in conv["events"][:ev_idx]:
        if e["role"] == "assistant":
            if e["content"]:
                lines.append(f"Agent: {e['content']}")
            for t in e["tool_calls"]:
                lines.append(f"Agent calls {t['name']}({t['raw']})")
        elif e["role"] == "user" and not e["tool_calls"] and e["content"]:
            lines.append(f"Customer: {e['content']}")
        elif e["role"] == "tool" and e["requestor"] == "assistant":
            lines.append(f"Tool result: {e['content'][:CAP]}")
    return "\n".join(lines)

def verify_prompt(it, variant, with_state):
    c, call = it["conv"], it["call"]
    doc = DOM[c["domain"]]["tool_docs"][call["name"]]
    s = (f"Conversation so far:\n{transcript(c, c['log'][it['i']]['event'])}\n\n"
         f"Action to verify: {call['name']}({json.dumps(call['arguments'])})\n"
         f"Tool description: {doc['description']}\n"
         f"Tool output: {it['output']}")
    if with_state:
        v = it["variants"][variant]
        s += ("\n\nDatabase records that the call names or that changed, before the call:\n"
              f"{json.dumps(v['before'], indent=1)}\n\nThe same records after the call:\n{json.dumps(v['after'], indent=1)}")
    return s + "\n\nDid the action succeed?"

VARIANTS = ("correct", "noop", "wrong_record", "wrong_field", "wrong_value")
ROWS = []
for k, it in enumerate(ITEMS):
    for v in VARIANTS:
        if v in it["variants"]:
            ROWS.append({"id": f"{it['conv']['id']}|{it['i']}|{v}", "item": k, "task": it["conv"]["key"],
                         "domain": it["conv"]["domain"], "tool": it["call"]["name"], "variant": v,
                         "kind": v, "failed": v != "correct"})
VER = {}
for m, with_state in (("llm_output", False), ("llm_state", True)):
    VER[m] = llm.run(f"nb28_verify_{'state' if with_state else 'output'}",
                     {r["id"]: LLM.body([{"role": "system", "content": VERIFY_SYS},
                                         {"role": "user", "content": verify_prompt(ITEMS[r["item"]], r["variant"], with_state)}],
                                        max_tokens=200, schema=Verdict) for r in ROWS}, parse=Verdict)
    print(f"{m}: {len(ROWS)} calls, {sum(x['parsed'] is None for x in VER[m].values())} unparsed")

[nb28_verify_output] 1,706 new requests -> batch (0 cached)


[nb28_verify_output] uploaded input file 1/1 (1,706 requests, 20 MB)


[nb28_verify_output] started batch batch_6af2fa0fbe348190b050a832feb2dc0c


[nb28_verify_output] 1 min: 1 running, 0 queued, 0 results cached


[nb28_verify_output] 6 min: 1 running, 0 queued, 0 results cached


[nb28_verify_output] 11 min: 1 running, 0 queued, 0 results cached


... (43 outputs left out)

[nb28_verify_state] batch batch_6af32791edb4819093fbdac860981370 completed: 1,706 ok, 0 failed


[nb28_verify_state] 198 min: 0 running, 0 queued, 1,706 results cached


[nb28_verify_state] done in 197.6 min


llm_state: 1706 calls, 0 unparsed


## Scores

A verifier **flags** an action when it judges it not successful (an unparsed LLM verdict counts as not flagged; the
count is printed above). Precision, recall and F1 are computed over all variants pooled; intervals resample tasks.

In [12]:
METHODS = [("llm_output", "LLM, tool output (1)"), ("llm_state", "LLM, before/after state (2)"),
           ("graph", "graph check, hand patterns (3)"), ("graph_llm", "graph check, LLM patterns (3b)")]
for r in ROWS:
    it = ITEMS[r["item"]]
    view, args = it["variants"][r["variant"]], it["call"]["arguments"] or {}
    r["flag"] = {"llm_output": (VER["llm_output"][r["id"]]["parsed"] or {}).get("succeeded") is False,
                 "llm_state": (VER["llm_state"][r["id"]]["parsed"] or {}).get("succeeded") is False,
                 "graph": not check(PATTERNS[r["tool"]], args, view)[0]}
    lp = LLM_PATTERNS.get(r["tool"])
    try:
        r["flag"]["graph_llm"] = (not check(lp, args, view)[0]) if lp else False
    except Exception:
        r["flag"]["graph_llm"] = True                       # a pattern the checker cannot run flags everything

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    rc = tp / (tp + fn) if tp + fn else 0.0
    return p, rc, (2 * p * rc / (p + rc) if p + rc else 0.0)

def counts(rows, m):
    tp = sum(r["failed"] and r["flag"][m] for r in rows); fp = sum((not r["failed"]) and r["flag"][m] for r in rows)
    fn = sum(r["failed"] and not r["flag"][m] for r in rows)
    return tp, fp, fn

TASKS_ = sorted({r["task"] for r in ROWS})
def task_counts(rows, m):
    a = np.zeros((len(TASKS_), 3)); ix = {t: i for i, t in enumerate(TASKS_)}
    for r in rows:
        a[ix[r["task"]]] += [r["failed"] and r["flag"][m], (not r["failed"]) and r["flag"][m], r["failed"] and not r["flag"][m]]
    return a

def f1v(a):                                              # rows of (tp, fp, fn) summed -> F1, vectorised
    tp, fp, fn = a[..., 0], a[..., 1], a[..., 2]
    return np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)

BOOT = np.random.default_rng(7).integers(0, len(TASKS_), (5000, len(TASKS_))) if TASKS_ else None
def boot_f1(rows, m, base=None):
    a = task_counts(rows, m)[BOOT].sum(1)
    if base is None:
        return np.percentile(f1v(a), [2.5, 97.5])
    d = f1v(a) - f1v(task_counts(rows, base)[BOOT].sum(1))
    return np.percentile(d, [2.5, 97.5])

SUMMARY = {}
def table(label, rows):
    print(f"\n{label}  ({len({r['item'] for r in rows})} write actions, {len(rows)} variants, "
          f"{sum(r['failed'] for r in rows)} failed)")
    print(f"{'verifier':<34}{'precision':>10}{'recall':>8}{'F1':>7}{'F1 95% CI':>18}{'false alarms':>14}")
    for m, lab in METHODS:
        p, rc, f = prf(*counts(rows, m))
        lo, hi = boot_f1(rows, m)
        fa = np.mean([r["flag"][m] for r in rows if not r["failed"]]) if any(not r["failed"] for r in rows) else float("nan")
        SUMMARY[(label, m)] = (p, rc, f, lo, hi, fa)
        print(f"{lab:<34}{p:>10.1%}{rc:>8.1%}{f:>7.2f}{f'[{lo:.2f}, {hi:.2f}]':>18}{fa:>14.1%}")

table("all domains", ROWS)
for d in DOM:
    if any(r["domain"] == d for r in ROWS):
        table(d, [r for r in ROWS if r["domain"] == d])


all domains  (398 write actions, 1706 variants, 1308 failed)
verifier                           precision  recall     F1         F1 95% CI  false alarms


LLM, tool output (1)                   75.7%   24.1%   0.37      [0.31, 0.42]         25.4%
LLM, before/after state (2)            96.5%   98.2%   0.97      [0.97, 0.98]         11.6%
graph check, hand patterns (3)        100.0%   85.5%   0.92      [0.91, 0.93]          0.0%
graph check, LLM patterns (3b)         83.2%   95.9%   0.89      [0.88, 0.90]         63.6%

airline  (55 write actions, 275 variants, 220 failed)
verifier                           precision  recall     F1         F1 95% CI  false alarms
LLM, tool output (1)                   75.8%   11.4%   0.20      [0.06, 0.33]         14.5%
LLM, before/after state (2)            96.9%   98.6%   0.98      [0.96, 0.99]         12.7%
graph check, hand patterns (3)        100.0%   70.5%   0.83      [0.79, 0.87]          0.0%
graph check, LLM patterns (3b)         84.8%   85.9%   0.85      [0.82, 0.87]         61.8%

retail  (216 write actions, 891 variants, 675 failed)
verifier                           precision  recall     F1   

### Primary comparison and recall by failure kind

In [13]:
PRIMARY = {}
print(f"{'comparison (F1 difference)':<46}{'diff':>7}{'95% CI (tasks resampled)':>28}")
for a, b in (("graph", "llm_state"), ("graph", "llm_output"), ("llm_state", "llm_output"), ("graph_llm", "graph"),
             ("graph_llm", "llm_state")):
    d = prf(*counts(ROWS, a))[2] - prf(*counts(ROWS, b))[2]
    lo, hi = boot_f1(ROWS, a, base=b)
    PRIMARY[(a, b)] = (d, lo, hi)
    print(f"{a + ' - ' + b:<46}{d:>+7.2f}{f'[{lo:+.2f}, {hi:+.2f}]':>28}"
          f"{'  <- primary' if (a, b) == ('graph', 'llm_state') else ''}")

print(f"\n{'share flagged':<16}" + "".join(f"{lab.split('(')[-1][:-1]:>8}" for _, lab in METHODS) + "      n")
for k in ("correct", "noop", "wrong_record", "wrong_field", "wrong_value"):
    rs = [r for r in ROWS if r["kind"] == k]
    if rs:
        print(f"{k:<16}" + "".join(f"{np.mean([r['flag'][m] for r in rs]):>8.0%}" for m, _ in METHODS) + f"{len(rs):>7}")
print("(for 'correct' the share flagged is the false-alarm rate; for the failure kinds it is the recall)")

comparison (F1 difference)                       diff    95% CI (tasks resampled)


graph - llm_state                               -0.05              [-0.07, -0.04]  <- primary


graph - llm_output                              +0.56              [+0.50, +0.61]
llm_state - llm_output                          +0.61              [+0.55, +0.66]
graph_llm - graph                               -0.03              [-0.04, -0.02]
graph_llm - llm_state                           -0.08              [-0.09, -0.07]

share flagged          1       2       3      3b      n
correct              25%     12%      0%     64%    398
noop                 24%    100%    100%    100%    398
wrong_record         22%    100%    100%    100%    344
wrong_field          35%    100%     86%     96%    225
wrong_value          19%     94%     53%     87%    341
(for 'correct' the share flagged is the false-alarm rate; for the failure kinds it is the recall)


### One action, its variants, four verdicts

In [14]:
it = next((x for x in ITEMS if len(x["variants"]) >= 4 and x["conv"]["domain"] == "retail"), ITEMS[0] if ITEMS else None)
if it:
    print(f"{it['conv']['id']}: {it['call']['name']}({json.dumps(it['call']['arguments'])})\nTool output: {it['output'][:300]}")
    for r in [r for r in ROWS if r["item"] == ITEMS.index(it)]:
        print(f"\n[{r['kind']}]  " + "  ".join(f"{m}: {'FLAG' if r['flag'][m] else 'ok'}" for m, _ in METHODS))
        for m in ("llm_output", "llm_state"):
            print(f"   {m} reason: {(VER[m][r['id']]['parsed'] or {}).get('reason', '(unparsed)')}")
        print(f"   graph: {check(PATTERNS[r['tool']], it['call']['arguments'], it['variants'][r['variant']])[1] or 'pattern matched'}")

retail|0: exchange_delivered_order_items({"order_id": "#W2378156", "item_ids": ["1151293680", "4983901480"], "new_item_ids": ["7706410293", "7747408585"], "payment_method_id": "credit_card_9513926"})
Tool output: {"order_id": "#W2378156", "user_id": "yusuf_rossi_9620", "address": {"address1": "763 Broadway", "address2": "Suite 135", "city": "Philadelphia", "country": "USA", "state": "PA", "zip": "19122"}, "items": [{"name": "Headphones", "product_id": "6992792935", "item_id": "4202497723", "price": 342.81, "

[correct]  llm_output: ok  llm_state: ok  graph: ok  graph_llm: FLAG
   llm_output reason: The order now shows an exchange request for exactly the two specified items and replacement variants, using the confirmed payment method, with no unrelated changes shown.
   llm_state reason: The order now records the requested two item IDs, their replacement variants, the original payment method, and the calculated price difference, with no other fields changed.
   graph: pattern matched

[

## What this run cost, and a forecast of the full run

The recording is the expensive part (the agent's prompt carries the whole policy at every turn). The forecast scales the
measured tokens per conversation (per domain) and per verifier call to 278 tasks, at the Batch price (each round is one
Batch job) and at the standard price.

In [15]:
STAGES = sorted(f[:-6] for f in os.listdir(llm.cache) if f.startswith("nb28_") and f.endswith(".jsonl"))
LEDGER = llm.ledger(STAGES)
rec = [s for s in STAGES if s.startswith("nb28_rec_")]
agg = lambda ss, k: sum(LEDGER[s][k] for s in ss)
print(f"{'stage':<26}{'calls':>8}{'input tok':>13}{'output tok':>12}{'USD':>8}{'via Batch':>11}")
for name, ss in (("recording (all rounds)", rec), *[(s, [s]) for s in STAGES if s not in rec]):
    print(f"{name:<26}{agg(ss, 'calls'):>8,}{agg(ss, 'in'):>13,}{agg(ss, 'out'):>12,}{agg(ss, 'usd'):>8.2f}"
          f"{np.mean([LEDGER[s]['batch_share'] for s in ss]):>11.0%}")
COST = sum(LEDGER[s]["usd"] for s in STAGES)
print(f"{'TOTAL':<26}{agg(STAGES, 'calls'):>8,}{agg(STAGES, 'in'):>13,}{agg(STAGES, 'out'):>12,}{COST:>8.2f}")

FULL_N = Counter(t["domain"] for t in META["tasks"])
price = lambda i, o, tier: (i * {"batch": 0.05, "standard": 0.10}[tier] + o * {"batch": 0.25, "standard": 0.50}[tier]) / 1e6
fc = {"batch": 0.0, "standard": 0.0}; calls = 0
for d in DOM:
    cs = [c for c in ALL if c["domain"] == d]
    if not cs:
        continue
    i = np.mean([c["tokens"]["agent_in"] + c["tokens"]["user_in"] for c in cs]) * FULL_N[d] * len(SEEDS)
    o = np.mean([c["tokens"]["agent_out"] + c["tokens"]["user_out"] for c in cs]) * FULL_N[d] * len(SEEDS)
    calls += np.mean([c["llm_calls"] for c in cs]) * FULL_N[d] * len(SEEDS)
    for t in fc:
        fc[t] += price(i, o, t)
rec_fc = dict(fc)
per_task_rows = len(ROWS) / max(len(KEYS), 1)
for s in ("nb28_verify_output", "nb28_verify_state"):
    a = LEDGER[s]; n = per_task_rows * len(TASK)
    calls += n
    for t in fc:
        fc[t] += price(a["in"] / a["calls"] * n, a["out"] / a["calls"] * n, t)
print(f"\nExpected full-run n (scaled from this run's {len(KEYS)} tasks): ~{len(ITEMS) / len(KEYS) * len(TASK):,.0f} distinct "
      f"write actions, ~{per_task_rows * len(TASK):,.0f} variants")
print(f"Forecast, full run (278 tasks x {len(SEEDS)} passes): ~{calls:,.0f} calls; recording ${rec_fc['batch']:.2f} via Batch "
      f"(${rec_fc['standard']:.2f} standard); total ${fc['batch']:.2f} via Batch, ${fc['standard']:.2f} standard.")
print("cost_forecast.py, '5.4 offline': 16,680 calls, $6.63 (recording $4.90 at the standard tier, verifiers via Batch).")

stage                        calls    input tok  output tok     USD  via Batch
recording (all rounds)      15,349   72,828,532     653,630    3.96        46%
nb28_pattern_llm                19       37,324       1,850    0.00       100%
nb28_verify_output           1,706    5,272,782      84,002    0.28       100%
nb28_verify_state            1,706    7,392,806      83,485    0.39       100%
TOTAL                       18,780   85,531,444     822,967    4.64

Expected full-run n (scaled from this run's 278 tasks): ~398 distinct write actions, ~1,706 variants
Forecast, full run (278 tasks x 3 passes): ~18,761 calls; recording $3.80 via Batch ($7.61 standard); total $4.48 via Batch, $8.96 standard.
cost_forecast.py, '5.4 offline': 16,680 calls, $6.63 (recording $4.90 at the standard tier, verifiers via Batch).


## Limitations

- **Synthetic faults.** No-ops and wrong changes are injected into the recorded state; real tool bugs may look different
  (partial writes, delayed writes). The wrong-change generator and the patterns come from the same author, so the graph
  check's recall on wrong changes is an upper bound for patterns of this detail; the LLM-written patterns (3b) and the
  recall per failure kind show how much it depends on the author.
- **What "correct" means.** The correct variant is the call executed as requested. Whether the agent should have made the
  call (policy) is τ²-bench's task reward, a different question; verifier (2) may flag a correct execution of a bad call.
- **Offline replay.** The verifiers judge recorded actions; the effect of a detection on the rest of the conversation
  (retry, telling the user) is not measured (PLAN §5a: final task success is not part of 5.4).
- **Shown state.** The records shown to (2) and (3) are those the call names plus those that changed in the variant;
  long maps (flight dates, product variants) are cut to the entries that differ or that the call names. A verifier that
  must find the changed record in the whole database would face a harder task.
- **Patterns check what their author thought of.** A wrong value in a field no clause names (a refund amount, a seat
  count) passes the graph check; this is the main source of its misses (recall on `wrong_value`).
- **Passes are not independent.** The three recording passes replay the same tasks; repeated actions are removed, and
  the intervals resample whole tasks, but similar actions of one task still count several times.
- **One model** (`gpt-6-luna`, no reasoning) as agent, user simulator, verifier and pattern writer.

**References.** Barres et al., *τ²-Bench: Evaluating Conversational Agents in a Dual-Control Environment*, arXiv
2506.07982 (2025) · Zhu et al., *When Tools Fail* (ToolMaze), arXiv 2606.05806 (2026) · Sun et al., *Tools Fail:
Detecting Silent Errors in Faulty Tools*, arXiv 2406.19228 (2024) · Wu et al., *StructAgent*, arXiv 2607.11388 (2026) ·
Lù et al., *AgentRewardBench*, arXiv 2504.08942 (2025) · Reddy et al., *Reason Less, Verify More*, arXiv 2607.07405 (2026).

In [16]:
print("=" * 118)
print(f"RESULTS SUMMARY — Notebook 4.4: outcome verification on τ²-bench (commit {TAU2_COMMIT[:7]}), {MODE} run, gpt-6-luna")
print("=" * 118)
print(f"Recorded tasks: {len(KEYS)} x {len(SEEDS)} passes ({dict(Counter(c['domain'] for c in REC))}); task reward of the "
      f"recordings (state-based): {np.mean([c['reward'] for c in ALL]):.0%}")
print(f"Distinct write actions verified: {len(ITEMS)} x up to 5 variants = {len(ROWS)} items ({sum(r['failed'] for r in ROWS)} failed)")
print(f"\n{'verifier':<34}{'precision':>10}{'recall':>8}{'F1':>7}{'F1 95% CI':>18}{'false alarms':>14}")
for m, lab in METHODS:
    p, rc, f, lo, hi, fa = SUMMARY[("all domains", m)]
    print(f"{lab:<34}{p:>10.1%}{rc:>8.1%}{f:>7.2f}{f'[{lo:.2f}, {hi:.2f}]':>18}{fa:>14.1%}")
d, lo, hi = PRIMARY[("graph", "llm_state")]
verdict = "graph check better" if lo > 0 else "graph check worse" if hi < 0 else "no resolvable difference"
print(f"\nPRIMARY  F1(graph check) - F1(LLM with the same state) = {d:+.2f}  [{lo:+.2f}, {hi:+.2f}]  ->  {verdict}")
print(f"Patterns: {len(PATTERNS)} hand-written (all write tools); LLM-written variant: F1 "
      f"{SUMMARY[('all domains', 'graph_llm')][2]:.2f} vs. {SUMMARY[('all domains', 'graph')][2]:.2f} hand-written")
print(f"Cost of this run: ${COST:.2f}")
print("\nLecture message (Task 4.4): an action's success should be checked against the state, not taken from the tool's"
      "\nreply; whether a graph pattern does that better than an LLM shown the same state is the PRIMARY line.")

RESULTS SUMMARY — Notebook 4.4: outcome verification on τ²-bench (commit 96a3ef9), batch run, gpt-6-luna
Recorded tasks: 278 x 3 passes ({'airline': 50, 'retail': 114, 'telecom': 114}); task reward of the recordings (state-based): 49%
Distinct write actions verified: 398 x up to 5 variants = 1706 items (1308 failed)

verifier                           precision  recall     F1         F1 95% CI  false alarms
LLM, tool output (1)                   75.7%   24.1%   0.37      [0.31, 0.42]         25.4%
LLM, before/after state (2)            96.5%   98.2%   0.97      [0.97, 0.98]         11.6%
graph check, hand patterns (3)        100.0%   85.5%   0.92      [0.91, 0.93]          0.0%
graph check, LLM patterns (3b)         83.2%   95.9%   0.89      [0.88, 0.90]         63.6%

PRIMARY  F1(graph check) - F1(LLM with the same state) = -0.05  [-0.07, -0.04]  ->  graph check worse
Patterns: 19 hand-written (all write tools); LLM-written variant: F1 0.89 vs. 0.92 hand-written
Cost of this run: $4.6